# This notebook shows you how to generate your own DBOF


In [1]:
import matplotlib.pyplot as plt
import numpy as np

# Initial set up
These steps help you build the project on a single machine. If you are using nrp Jupyterhub, it is recommended you use the next block instead.

#### Build the project
- ```pip install .```

#### Install aws cli (optional)
You will want to install this if you want to manually see the data stored in the s3 bucket.

Example for installing on linux :
- ```sudo curl "https://awscli.amazonaws.com/awscli-exe-linux-x86_64.zip" -o "awscliv2.zip"```
- ```sudo unzip awscliv2.zip```
- ```sudo ./aws/install```

#### Set your aws secret and access keys
If you are accessing or writing data to S3, you must set credentials.
NOTE : S3 is all that is supported currently.
This typically corresponds to an NRP S3 bucket.

windows:

- ```$env:AWS_ACCESS_KEY_ID="..."```
- ```$env:AWS_SECRET_ACCESS_KEY="..."```

unix:

- ```export AWS_ACCESS_KEY_ID=...```
- ```export AWS_SECRET_ACCESS_KEY=...```

# Dataset generation config (quick reference)

This job is fully controlled by a YAML config file. The config defines **what time range is scanned**, **how often snapshots are taken**, and **how many spatial patches are sampled per snapshot**.

### Temporal sampling
- `data.timestep_hours`
  Total time window (in hours) to scan starting from `start_record`.
  Example: `8760` = one LLC model year (336 days).

- `data.sampling_step`
  Spacing **in hours** between snapshots within the window.
  Example: `2190` with `timestep_hours=8760` → 4 evenly spaced snapshots.

- `data.start_record`
  First valid wind/forcing record (default: `1180`). You probably don't want to change this.

Each snapshot corresponds to a **single instantaneous model timestep** (not an average).

### Spatial sampling
- `sampling.sample_points_per_snapshot`
  Number of cutouts sampled per snapshot.

- `sampling.bias_to_high_gradients`
  Exponential bias favoring high-gradient regions when sampling.

### Patch geometry
- `output.target_km_res`
  Physical patch size in km. Default 150.

- `output.down_sample_res`
  Pixel resolution of the extracted cutouts.
  Must be small enough to safely resolve `target_km_res` on the LLC grid.
  Default 64

### Output / logging
- `output.bucket`, `output.folder`
  S3 location for dataset output.

- `run.run_id`
  Unique identifier for this run (used for logs and output paths).
  If the run_id you use exists on the write location (s3 bucket), your data will be appended to the existing data from previous run(s).

- `run.log_dir`
  Local directory where logs are written.

### Invariants (not configurable)
Grid topology, model cadence (144 timesteps/hour), and dataset offsets are fixed
LLC4320 constants and are enforced in code.

### Examples
See existing configs in configs/ for examples

# A note about logs
The run logs will be stored locally on your machine in the directory you specify.
- ```log_dir/run_id/```
However under the current logic, if you attempt to run the script and the specified log output path already exists, the script will fail to run. This is by design. The reasons are as follows.
- run_id is also used for the zarr dataset path. You likely do not want to send two different runs to the same dataset.
- You will likely not want to override your previous run logs.

If you want to override this logic, simply delete the existing log output path from your local machine. You can also override the run_id with a cli argument ```--run_id```

# Convenience block for running in NRP Jupyterhub

In [2]:
# This is for running this notebook on nrp Jupyterhub. Set True if you are on NRP Jupyterhub
RUNNING_ON_NRP = False

# This builds the project and installs dependencies much faster on jupyterhub than pip install .
# There are much better ways to do this but this notebook is meant to be accessible and easy to use
if (RUNNING_ON_NRP):
    %pip install -e ../. --no-deps

    %pip install xgcm
    %pip install zarr
    %pip install boto3
    %pip install ujson
    %pip install scikit-fmm

    # !sudo curl "https://awscli.amazonaws.com/awscli-exe-linux-x86_64.zip" -o "awscliv2.zip"
    # !sudo unzip awscliv2.zip
    # !sudo ./aws/install todo do we need aws installed for this notebook?

## Example run 1
- 1 year of data
- 4 time snapshots evenly spaced through the year
- 150 cutouts per timestamp

If you are running this on your laptop or pc expect it to take a while.

In [11]:
import datetime
run_id = f"year_4x150_{datetime.datetime.now(datetime.UTC).strftime('%Y%m%d_%H%M%S')}"
print(f"Your run id is: {run_id}")
print("Make sure and use the same run_d later when accessing your data")

Your run id is: year_4x150_20260129_044851
Make sure and use the same run_d later when accessing your data


In [3]:
# note: that --run_id can overwrite the config file
# note: All Dask logs are warnings. Don't be alarmed.
!generate-llc-dataset --config ../configs/example_1year_4_snapshot.yaml --run_id test01

2026-01-28 11:07:44,159 | INFO | Arguments parsed successfully. Logging set up. Running script.
2026-01-28 11:07:45,245 | INFO | Dask Client <Client: 'tcp://127.0.0.1:61213' processes=4 threads=12, memory=31.10 GiB>
2026-01-28 11:07:45,245 | INFO | Processing: [ 180288  495648  811008 1126368] time snapshots
s3://llc/native_grid_dbof_training_data/year_4x150/dataset_creation.zarr
2026-01-28 11:07:45,254 | INFO | Found credentials in shared credentials file: ~/.aws/credentials
2026-01-28 11:07:47,052 | INFO | Zarr dataset_creation created.
2026-01-28 11:07:47,052 | INFO | Fetching grid file
2026-01-28 11:08:27,163 | INFO | Calculating land and face masks
Opening 13 Kerchunk JSON files...
Parsing JSON metadata into Python dicts...
Creating lazy xarray datasets...
Computing delayed datasets...
Combining datasets by coordinates...
Dataset combined successfully.
2026-01-28 11:14:06,850 | INFO | Data loaded for iteration: 180288
2026-01-28 11:14:06,850 | INFO | Calculating ice mask
2026-01-2

C:\Users\Jake Tallman\PycharmProjects\dbof-in-native-grid\.venv1\Lib\site-packages\zarr\core\dtype\npy\bytes.py:386: UnstableSpecificationWarning: The data type (NullTerminatedBytes(length=32)) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change without warning in a future version of Zarr Python. Arrays stored with this data type may be unreadable by other Zarr libraries. Use this data type at your own risk! Check https://github.com/zarr-developers/zarr-extensions/tree/main/data-types for the status of data type specifications for Zarr V3.
  v3_unstable_dtype_warning(self)

  0%|          | 0/4 [00:00<?, ?it/s]C:\Users\Jake Tallman\PycharmProjects\dbof-in-native-grid\.venv1\Lib\site-packages\distributed\client.py:3370: UserWarning: Sending large graph of size 235.69 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph with

## Example run 2 - very small
- 2 weeks of data
- 1 snapshot per week
- 10 cutouts per timestamp

If you are running this on your laptop or pc expect it to take a while.

In [4]:
!generate-llc-dataset --config ../configs/test.yaml --run_id test01

2026-01-28 10:04:44,037 | INFO | Arguments parsed successfully. Logging set up. Running script.
2026-01-28 10:04:45,133 | INFO | Dask Client <Client: 'tcp://127.0.0.1:50087' processes=4 threads=12, memory=31.10 GiB>
2026-01-28 10:04:45,133 | INFO | Processing: [180288 204480] time snapshots
s3://llc/native_grid_dbof_training_data/test01/dataset_creation.zarr
2026-01-28 10:04:45,152 | INFO | Found credentials in shared credentials file: ~/.aws/credentials
2026-01-28 10:04:47,007 | INFO | Zarr dataset_creation created.
2026-01-28 10:04:47,007 | INFO | Fetching grid file
2026-01-28 10:04:55,665 | INFO | Calculating land and face masks
Opening 13 Kerchunk JSON files...
Parsing JSON metadata into Python dicts...
Creating lazy xarray datasets...
Computing delayed datasets...
Combining datasets by coordinates...
Dataset combined successfully.
2026-01-28 10:09:31,001 | INFO | Data loaded for iteration: 180288
2026-01-28 10:09:31,001 | INFO | Calculating ice mask
2026-01-28 10:10:02,807 | INFO 

C:\Users\Jake Tallman\PycharmProjects\dbof-in-native-grid\.venv1\Lib\site-packages\zarr\core\dtype\npy\bytes.py:386: UnstableSpecificationWarning: The data type (NullTerminatedBytes(length=32)) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change without warning in a future version of Zarr Python. Arrays stored with this data type may be unreadable by other Zarr libraries. Use this data type at your own risk! Check https://github.com/zarr-developers/zarr-extensions/tree/main/data-types for the status of data type specifications for Zarr V3.
  v3_unstable_dtype_warning(self)

  0%|          | 0/2 [00:00<?, ?it/s]C:\Users\Jake Tallman\PycharmProjects\dbof-in-native-grid\.venv1\Lib\site-packages\distributed\client.py:3370: UserWarning: Sending large graph of size 235.69 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph with

In [ ]:
# !pip install s3fs
# aws --endpoint https://s3-west.nrp-nautilus.io s3 ls s3://llc/native_grid_dbof_training_data/script_test_00/ --human-readable
#
# aws --endpoint https://s3-west.nrp-nautilus.io s3 rm s3://llc/native_grid_dbof_training_data/script_test_00/ --recursive --dryrun